# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² clinical colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print out dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The `@id` fields uniquely identify each entity within the dataset and are essential for unambiguous referencing.

In [ ]:
# List all record sets in the dataset by `@id` and print their fields

record_sets = dataset.record_sets  # List of RecordSet schema objects
print("Found record sets:")
record_set_ids = []
for rs in record_sets:
    print(f"  - @id: {rs.id} | name: {getattr(rs, 'name', None)}")
    record_set_ids.append(rs.id)
    print("    Fields and columns:")
    for fld in rs.fields:
        print(f"      * Field @id: {fld.id} | name: {getattr(fld, 'name', None)} | dataType: {getattr(fld, 'data_type', None)}")
        if fld.columns:
            for col in fld.columns:
                print(f"          - Column @id: {col.id} | name: {getattr(col, 'name', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references use their `@id` values.

In [ ]:
# Extract data for all record sets
dataframes = {}

for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    dataframes[rset_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rset_id])} records for record set: {rset_id}")
    if len(dataframes[rset_id]) > 0:
        print(f"  Fields: {list(dataframes[rset_id].columns)}")

# For demonstration, select the first record set if multiple exist
if len(record_set_ids) > 0:
    main_recordset_id = record_set_ids[0]  # Change if you want a specific set
    df = dataframes[main_recordset_id]
    print(f"\nFirst few rows of record set {main_recordset_id}:")
    display(df.head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will choose one numeric field and one group field from the main record set (using their `@id`).

In [ ]:
# Choose a numeric and a group field from the schema
# Inspect main_recordset_id's fields to select appropriate IDs

# Replace these with the actual field @ids from earlier overview
numeric_field_id = None
group_field_id = None

# Find a field with dataType Integer or Float as numeric field
main_rs = None
for rs in record_sets:
    if rs.id == main_recordset_id:
        main_rs = rs
        break

if main_rs:
    for f in main_rs.fields:
        dt = getattr(f, "data_type", None)
        if dt in ("schema:Integer", "schema:Float") and numeric_field_id is None:
            numeric_field_id = f.id
        if ("location" in f.id.lower() or "sex" in f.id.lower() or "msi" in f.id.lower() or dt=="schema:Text") and group_field_id is None:
            group_field_id = f.id
        if numeric_field_id and group_field_id:
            break

print(f"Chosen numeric field: {numeric_field_id}")
print(f"Chosen group field: {group_field_id}")

# Ensure field IDs are present in columns
df = dataframes[main_recordset_id]
if numeric_field_id is not None and numeric_field_id in df.columns:
    # Convert column to numeric (errors='coerce' will set invalid parsing as NaN)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].median()
    print(f"Applying filter: {numeric_field_id} > {threshold}")
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group if possible
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count','mean','std','min','max'])
        print(f"\nGrouped statistics by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields using the chosen numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if fields exist
if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Not enough information to plot. Check previous outputs for available fields.")

## 6. Conclusion
This exploration demonstrates how to load, process, and visualize a FAIR² dataset with `mlcroissant`, referencing all data entities by their schema `@id`. The notebook can be extended for more domain-specific analyses or modeling.